# 02 · SFT data → QLoRA → đánh giá base và LoRA

**Settings:** GPU **T4 x2** · Internet **On**

**Secrets** (Add-ons → Secrets), bật cho notebook này:
- `TEACHER_API_KEY`: API key của teacher LLM (dùng để tạo dữ liệu và làm judge)
- `HF_TOKEN` (không bắt buộc): tải model từ HuggingFace nhanh hơn

Thời gian ước tính:
- Tạo dữ liệu SFT: 20–60 phút (tuỳ giới hạn tốc độ của API)
- Train: 1–2 giờ
- Đánh giá: khoảng 40 phút

In [ ]:
REPO_URL = "https://github.com/sinhtruc24/Vietnamese-Legal-RAG.git"
WORK = "/kaggle/working/Vietnamese-Legal-RAG"

import os, socket
try:
    socket.create_connection(("github.com", 443), timeout=5).close()
except OSError:
    raise SystemExit("Không có Internet: Settings (bên phải) -> Internet = On, "
                     "rồi Run -> Restart & clear cell outputs và chạy lại.")

if not os.path.exists(WORK):
    !git clone -q {REPO_URL} {WORK}
else:
    !git -C {WORK} pull -q
assert os.path.exists(WORK), "Clone thất bại: kiểm tra REPO_URL và repo có để Public không"
%cd {WORK}
!git log --oneline -1

In [ ]:
!pip install -q -e ".[ml]" openai huggingface_hub
!pip install -q -r requirements-train.txt
# Kaggle cài sẵn torchao 0.10; peft mới từ chối bản này khi gắn LoRA vào model fp16 (ImportError).
# Dự án không dùng torchao nên gỡ đi.
!pip uninstall -y -q torchao

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["TEACHER_API_KEY"] = secrets.get_secret("TEACHER_API_KEY")
os.environ["JUDGE_API_KEY"] = os.environ["TEACHER_API_KEY"]
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception:
    pass

# ---- Teacher / judge: key và endpoint PHẢI cùng nhà cung cấp ----
# Gemini (key từ aistudio.google.com):
TEACHER_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
TEACHER_MODEL = "gemini-3.1-flash-lite"   # judge: 15 RPM, 500 request/ngày (free tier)
REFERENCE_MODEL = "gemini-3.5-flash-lite" # mốc tham chiếu ở mục 4, quota riêng
WORKERS = 4
RPM = 14            # Flash Lite free tier: 15 request/phút (xem ai.dev/rate-limit)
SFT_LIMIT = 1000    # free tier: 1000 câu mất khoảng 4 giờ; 0 = toàn bộ khoảng 2.4K câu
# Trả phí (OpenAI, hoặc Gemini đã bật billing): WORKERS = 8; RPM = 0; SFT_LIMIT = 0

STUDENT_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LORA_NAME = "lora_v2"            # phiên bản LoRA: adapter ở outputs/qwen-legal-{LORA_NAME}, kết quả ở results/gen_{LORA_NAME}*
REUSE_UPLOADED_ADAPTER = False   # True: dùng adapter upload sẵn trong Input thay vì train

## 1. Dữ liệu và BM25 index
Chỉ cần BM25 để lấy hard negative, build mất khoảng 1 phút.

In [ ]:
!python scripts/download_data.py
!python scripts/build_index.py --bigrams --skip-dense --index-dir indexes/full

## 2a. (Khuyến nghị) Dùng dữ liệu SFT đã tạo trên máy
Nếu đã tạo `data/sft/` trên máy và upload lên thành Kaggle Dataset (Add Input → chọn dataset đó), ô dưới sẽ tự tìm và chép vào.
Khi ô báo **"Dùng dữ liệu SFT có sẵn"** thì **bỏ qua mục 2b**, chuyển thẳng xuống mục 3.

In [ ]:
import glob, os, shutil
SFT_FROM_INPUT = False
found = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True),
               key=lambda p: -sum(1 for _ in open(p, encoding="utf-8")))  # nhiều dataset -> lấy bản nhiều mẫu nhất
if len(found) > 1:
    print("Có nhiều train.jsonl trong Input:", found)
if found:
    src = os.path.dirname(found[0])
    SFT_FROM_INPUT = True
    os.makedirs("data/sft", exist_ok=True)
    for name in ("train.jsonl", "val.jsonl", "raw.jsonl"):
        if os.path.exists(f"{src}/{name}"):
            shutil.copy(f"{src}/{name}", f"data/sft/{name}")
    n = sum(1 for _ in open("data/sft/train.jsonl", encoding="utf-8"))
    print(f"Dùng dữ liệu SFT có sẵn từ {src}: {n} mẫu train -> bỏ qua mục 2b")
else:
    print("Không có dataset SFT -> chạy mục 2b để tạo trên Kaggle")

## 2b. Tạo dữ liệu SFT trên Kaggle
Chạy thử 30 câu trước và **đọc kỹ vài mẫu**, rồi mới chạy toàn bộ.
Script chạy tiếp được nếu bị ngắt giữa chừng.

In [ ]:
if SFT_FROM_INPUT:
    print("Đã có dữ liệu SFT từ dataset -> bỏ qua")
else:
    !python scripts/build_sft_data.py --index-dir indexes/full --limit 30 --workers {WORKERS} --rpm {RPM} \
        --teacher-base-url {TEACHER_BASE_URL} --teacher-model {TEACHER_MODEL}

In [ ]:
import json
rows = [json.loads(l) for l in open("data/sft/raw.jsonl", encoding="utf-8")]
for r in rows[:3]:
    print("TYPE:", r["type"], "| gold:", r["gold"])
    print(r["prompt"][1]["content"][-300:])
    print("ANSWER:", r["completion"][0]["content"])
    print("=" * 100)

In [ ]:
if SFT_FROM_INPUT:
    print("Đã có dữ liệu SFT từ dataset -> bỏ qua")
else:
    # Chạy tiếp được: hết quota ngày hoặc session bị ngắt thì chạy lại đúng ô này, script làm tiếp phần còn thiếu
    !python scripts/build_sft_data.py --index-dir indexes/full --limit {SFT_LIMIT} --workers {WORKERS} --rpm {RPM} \
        --teacher-base-url {TEACHER_BASE_URL} --teacher-model {TEACHER_MODEL}

## 3. Train QLoRA
Dùng **cả 2 GPU** bằng DDP (`torchrun`), mỗi GPU một bản model, nhanh gần gấp đôi so với 1 GPU.
1 epoch là đủ (lần 1: eval_loss tốt nhất ở cuối epoch 1). Checkpoint lưu mỗi 25 bước vào `outputs/qwen-legal-{LORA_NAME}/`.

**Chạy nền (khuyến nghị):** bấm **Save Version → Save & Run All**. Kaggle chạy tối đa 12 giờ, không cần mở trình duyệt.

**Chia làm nhiều phiên:** nếu phiên trước bị ngắt giữa chừng, thêm Output của phiên đó làm Input
(Add Input → Your Work → notebook này). Ô dưới tự chép checkpoint về và train tiếp từ đó.

In [ ]:
# Nạp kết quả train từ phiên trước (Add Input -> Your Work -> output của notebook này)
#  - đã train xong (hoặc gần xong): lấy LoRA của checkpoint có eval_loss thấp nhất -> bỏ qua bước train
#  - đang dở: chép checkpoint về để train tiếp (--resume)
import glob, json, os, shutil

OUT = f"outputs/qwen-legal-{LORA_NAME}"
ADAPTER = f"{OUT}/adapter"

def step_of(path):
    return int(path.rsplit("-", 1)[1])

prev = sorted(glob.glob(f"/kaggle/input/**/qwen-legal-{LORA_NAME}/checkpoint-*", recursive=True), key=step_of)
uploaded = sorted(glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True))
if prev:
    os.makedirs(OUT, exist_ok=True)
    for ckpt in prev:
        dst = f"{OUT}/{os.path.basename(ckpt)}"
        if not os.path.exists(dst):
            shutil.copytree(ckpt, dst)
    state = json.load(open(f"{prev[-1]}/trainer_state.json"))
    done, total = int(state["global_step"]), int(state["max_steps"])
    if done >= 0.9 * total:  # last steps run at ~zero learning rate: nothing left to learn
        history = [h for h in state["log_history"] if "eval_loss" in h]
        best = min(history, key=lambda h: float(h["eval_loss"]))
        src = f"{OUT}/checkpoint-{int(best['step'])}"
        os.makedirs(ADAPTER, exist_ok=True)
        for f in glob.glob(f"{src}/adapter_*"):
            shutil.copy(f, ADAPTER)
        from transformers import AutoTokenizer
        AutoTokenizer.from_pretrained(STUDENT_MODEL).save_pretrained(ADAPTER)
        json.dump(history, open(f"{OUT}/eval_history.json", "w"), indent=2)
        print(f"Phiên trước đã train {done}/{total} bước -> dùng adapter từ {src} "
              f"(eval_loss={float(best['eval_loss']):.4f}), BỎ QUA bước train")
    else:
        print(f"Phiên trước dừng ở bước {done}/{total} -> train tiếp từ checkpoint-{done}")
elif REUSE_UPLOADED_ADAPTER and uploaded:
    src = os.path.dirname(uploaded[0])
    os.makedirs(ADAPTER, exist_ok=True)
    for f in glob.glob(f"{src}/*"):
        if os.path.basename(f).startswith(("adapter_", "tokenizer", "chat_template")):
            shutil.copy(f, ADAPTER)
    print(f"Dùng adapter đã upload từ {src} -> BỎ QUA bước train:", sorted(os.listdir(ADAPTER)))
else:
    if uploaded:
        print("Có adapter trong Input nhưng REUSE_UPLOADED_ADAPTER = False -> không dùng")
    print(f"Train {LORA_NAME} từ đầu")

In [ ]:
import os, torch
if os.path.exists(f"{OUT}/adapter/adapter_model.safetensors"):
    print("Đã có adapter -> bỏ qua bước train")
else:
    N_GPUS = torch.cuda.device_count()
    LAUNCH = f"torchrun --nproc_per_node {N_GPUS}" if N_GPUS > 1 else "python"
    print(f"{N_GPUS} GPU -> {LAUNCH}")
    !{LAUNCH} training/train_qlora.py --model {STUDENT_MODEL} \
        --train data/sft/train.jsonl --val data/sft/val.jsonl \
        --output-dir {OUT} --epochs 1 --resume

In [ ]:
# Đảm bảo có adapter: nếu bước train dừng trước khi lưu adapter/, lấy LoRA từ checkpoint có eval_loss thấp nhất
import glob, json, os, shutil
import pandas as pd

OUT = f"outputs/qwen-legal-{LORA_NAME}"
ADAPTER = f"{OUT}/adapter"
ckpts = sorted(glob.glob(f"{OUT}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
state = json.load(open(f"{ckpts[-1]}/trainer_state.json")) if ckpts else {}
history = [h for h in state.get("log_history", []) if "eval_loss" in h]

if not os.path.exists(f"{ADAPTER}/adapter_model.safetensors"):
    best = min(history, key=lambda h: float(h["eval_loss"]))
    src = f"{OUT}/checkpoint-{int(best['step'])}"
    os.makedirs(ADAPTER, exist_ok=True)
    for f in glob.glob(f"{src}/adapter_*"):
        shutil.copy(f, ADAPTER)
    from transformers import AutoTokenizer
    AutoTokenizer.from_pretrained(STUDENT_MODEL).save_pretrained(ADAPTER)
    json.dump(history, open(f"{OUT}/eval_history.json", "w"), indent=2)
    print(f"Lấy adapter từ {src} (eval_loss={float(best['eval_loss']):.4f})")

print(sorted(os.listdir(ADAPTER)))
pd.DataFrame(json.load(open(f"{OUT}/eval_history.json")))[["step", "eval_loss", "eval_mean_token_accuracy"]]

## 4. Đánh giá: base, LoRA và teacher
Mọi mô hình nhận **cùng ngữ cảnh** (chế độ oracle, seed cố định), nên chênh lệch điểm chỉ phản ánh chất lượng của mô hình.

In [ ]:
import glob, os, shutil
# Tái sử dụng kết quả đã có từ phiên trước (Add Input -> Your Work): chỉ chạy phần còn thiếu
os.makedirs("results", exist_ok=True)
for f in glob.glob("/kaggle/input/**/gen_*", recursive=True):
    dst = f"results/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst)

def done(name):
    return os.path.exists(f"results/gen_{name}_summary.json")

JUDGE = f"--judge-base-url {TEACHER_BASE_URL} --judge-model {TEACHER_MODEL} --rpm {RPM}"
LIMIT = 150   # base + LoRA: khoảng 240 lượt judge, vừa quota 500/ngày của TEACHER_MODEL

if done("base"):
    print("base: đã có kết quả -> bỏ qua")
else:
    !python scripts/eval_generation.py --backend hf --llm-model {STUDENT_MODEL} --name base --limit {LIMIT} {JUDGE}

if done(LORA_NAME):
    print(f"{LORA_NAME}: đã có kết quả -> bỏ qua")
else:
    !python scripts/eval_generation.py --backend hf --llm-model {STUDENT_MODEL} \
        --adapter {OUT}/adapter --name {LORA_NAME} --limit {LIMIT} {JUDGE}

In [ ]:
# Teacher làm mốc tham chiếu (gọi qua API)
os.environ["LLM_API_KEY"] = os.environ["TEACHER_API_KEY"]
if done("teacher"):
    print("teacher: đã có kết quả -> bỏ qua")
else:
    !python scripts/eval_generation.py --llm-base-url {TEACHER_BASE_URL} --llm-model {REFERENCE_MODEL} \
        --name teacher --limit {LIMIT} {JUDGE}

In [ ]:
import glob
summaries = [json.load(open(p)) for p in sorted(glob.glob("results/gen_*_summary.json"))]
cols = ["name", "answer_rate", "gold_cite_rate", "cite_precision", "invalid_cite_rate",
        "uncited_rate", "refusal_acc", "faithfulness", "latency_ms"]
pd.DataFrame(summaries)[cols].set_index("name")

## 5. Phân tích lỗi
Xem những câu LoRA trả lời sai để viết phần phân tích trong README. Nhà tuyển dụng thường đánh giá cao phần này.

In [ ]:
lora = [json.loads(l) for l in open(f"results/gen_{LORA_NAME}.jsonl", encoding="utf-8")]
bad = [r for r in lora if (r["answerable"] and not r["cites_gold"]) or (not r["answerable"] and not r["refused"])]
print(f"{len(bad)} / {len(lora)} lỗi")
for r in bad[:5]:
    print("gold:", r["gold"], "| cited:", r["cited"], "| refused:", r["refused"])
    print(r["answer"][:500])
    print("-" * 100)

## 6. Lưu
**Save Version → Save & Run All**. Sau đó ở tab Output, tải về:
- `outputs/qwen-legal-lora/adapter/`: LoRA adapter, dùng cho vLLM/Docker
- `results/`: kết quả để điền README